In [1]:
import numpy as np
import pandas as pd
from matplotlib import pyplot
df=pd.read_csv(r"AI_dataset.csv")
df

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
0,1,CA-2017-152156,08-11-2017,11-11-2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600
1,2,CA-2017-152156,08-11-2017,11-11-2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400
2,3,CA-2017-138688,12-06-2017,16-06-2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200
3,4,US-2016-108966,11-10-2016,18-10-2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775
4,5,US-2016-108966,11-10-2016,18-10-2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9795,9796,CA-2017-125920,21-05-2017,28-05-2017,Standard Class,SH-19975,Sally Hughsby,Corporate,United States,Chicago,Illinois,60610.0,Central,OFF-BI-10003429,Office Supplies,Binders,"Cardinal HOLDit! Binder Insert Strips,Extra St...",3.7980
9796,9797,CA-2016-128608,12-01-2016,17-01-2016,Standard Class,CS-12490,Cindy Schnelling,Corporate,United States,Toledo,Ohio,43615.0,East,OFF-AR-10001374,Office Supplies,Art,"BIC Brite Liner Highlighters, Chisel Tip",10.3680
9797,9798,CA-2016-128608,12-01-2016,17-01-2016,Standard Class,CS-12490,Cindy Schnelling,Corporate,United States,Toledo,Ohio,43615.0,East,TEC-PH-10004977,Technology,Phones,GE 30524EE4,235.1880
9798,9799,CA-2016-128608,12-01-2016,17-01-2016,Standard Class,CS-12490,Cindy Schnelling,Corporate,United States,Toledo,Ohio,43615.0,East,TEC-PH-10000912,Technology,Phones,Anker 24W Portable Micro USB Car Charger,26.3760


In [18]:
print(df['Sub-Category'].unique())

<StringArray>
[  'Bookcases',      'Chairs',      'Labels',      'Tables',     'Storage',
 'Furnishings',         'Art',      'Phones',     'Binders',  'Appliances',
       'Paper', 'Accessories',   'Envelopes',   'Fasteners',    'Supplies',
    'Machines',     'Copiers']
Length: 17, dtype: str


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9800 entries, 0 to 9799
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9800 non-null   int64  
 1   Order ID       9800 non-null   str    
 2   Order Date     9800 non-null   str    
 3   Ship Date      9800 non-null   str    
 4   Ship Mode      9800 non-null   str    
 5   Customer ID    9800 non-null   str    
 6   Customer Name  9800 non-null   str    
 7   Segment        9800 non-null   str    
 8   Country        9800 non-null   str    
 9   City           9800 non-null   str    
 10  State          9800 non-null   str    
 11  Postal Code    9789 non-null   float64
 12  Region         9800 non-null   str    
 13  Product ID     9800 non-null   str    
 14  Category       9800 non-null   str    
 15  Sub-Category   9800 non-null   str    
 16  Product Name   9800 non-null   str    
 17  Sales          9800 non-null   float64
dtypes: float64(2), int6

In [3]:
df.isnull().sum()

Row ID            0
Order ID          0
Order Date        0
Ship Date         0
Ship Mode         0
Customer ID       0
Customer Name     0
Segment           0
Country           0
City              0
State             0
Postal Code      11
Region            0
Product ID        0
Category          0
Sub-Category      0
Product Name      0
Sales             0
dtype: int64

In [4]:
df.duplicated().sum()

np.int64(0)

In [20]:
weekly = (df.groupby(['Sub-Category', pd.Grouper(key='Order Date', freq='W')])['Sales']
            .sum().reset_index())
weekly.columns = ['sub_category', 'date', 'sales']
weekly

,sub_category,date,sales
0,Accessories,2015-01-11,31.200
1,Accessories,2015-01-18,796.690
2,Accessories,2015-02-01,468.900
3,Accessories,2015-02-08,115.360
4,Accessories,2015-02-15,474.420
...,...,...,...
2681,Tables,2018-12-02,4104.942
2682,Tables,2018-12-09,2422.312
2683,Tables,2018-12-16,3893.045
2684,Tables,2018-12-23,2136.826


In [22]:
full = []
for s, g in weekly.groupby('sub_category'):
    g = g.set_index('date')['sales'].asfreq('W', fill_value=0).reset_index()
    g['sub_category'] = s
    full.append(g)

weekly = pd.concat(full, ignore_index=True)

print(weekly['sub_category'].nunique(), weekly.shape)

17 (3503, 3)


In [25]:
cutoff = weekly['date'].max() - pd.Timedelta(weeks=12)
train = weekly[weekly['date'] <= cutoff]
test  = weekly[weekly['date'] >  cutoff]
test

,date,sales,sub_category
196,2018-10-14,2440.954,Accessories
197,2018-10-21,46.928,Accessories
198,2018-10-28,125.366,Accessories
199,2018-11-04,3143.032,Accessories
200,2018-11-11,2654.628,Accessories
...,...,...,...
3498,2018-12-02,4104.942,Tables
3499,2018-12-09,2422.312,Tables
3500,2018-12-16,3893.045,Tables
3501,2018-12-23,2136.826,Tables


In [27]:
from prophet import Prophet
import numpy as np

prophet_models = {}
prophet_test_pred = []

for s in weekly['sub_category'].unique():
    tr = train[train['sub_category'] == s][['date', 'sales']].rename(columns={'date': 'ds', 'sales': 'y'})
    te = test[test['sub_category'] == s][['date']].rename(columns={'date': 'ds'})

    m = Prophet()
    m.fit(tr)
    prophet_models[s] = m

    p = m.predict(te)[['ds', 'yhat']]
    p['yhat'] = p['yhat'].clip(lower=0)
    p['sub_category'] = s
    prophet_test_pred.append(p)

prophet_test_pred = pd.concat(prophet_test_pred)

10:30:28 - cmdstanpy - INFO - Chain [1] start processing
10:30:28 - cmdstanpy - INFO - Chain [1] done processing
10:30:29 - cmdstanpy - INFO - Chain [1] start processing
10:30:29 - cmdstanpy - INFO - Chain [1] done processing
10:30:29 - cmdstanpy - INFO - Chain [1] start processing
10:30:29 - cmdstanpy - INFO - Chain [1] done processing
10:30:29 - cmdstanpy - INFO - Chain [1] start processing
10:30:29 - cmdstanpy - INFO - Chain [1] done processing
10:30:29 - cmdstanpy - INFO - Chain [1] start processing
10:30:29 - cmdstanpy - INFO - Chain [1] done processing
10:30:29 - cmdstanpy - INFO - Chain [1] start processing
10:30:29 - cmdstanpy - INFO - Chain [1] done processing
10:30:29 - cmdstanpy - INFO - Chain [1] start processing
10:30:29 - cmdstanpy - INFO - Chain [1] done processing
10:30:29 - cmdstanpy - INFO - Chain [1] start processing
10:30:30 - cmdstanpy - INFO - Chain [1] done processing
10:30:30 - cmdstanpy - INFO - Chain [1] start processing
10:30:30 - cmdstanpy - INFO - Chain [1]

In [28]:
from xgboost import XGBRegressor

def add_clues(w):
    w = w.sort_values(['sub_category', 'date']).copy()
    g = w.groupby('sub_category')['sales']
    w['week_no'] = w['date'].dt.isocalendar().week.astype(int)
    w['lag1'] = g.shift(1)
    w['lag4'] = g.shift(4)
    w['avg4'] = g.transform(lambda x: x.shift(1).rolling(4).mean())
    w['sub_code'] = w['sub_category'].astype('category').cat.codes
    return w

feat = add_clues(weekly).dropna()
cols = ['sub_code', 'week_no', 'lag1', 'lag4', 'avg4']

tr = feat[feat['date'] <= cutoff]
te = feat[feat['date'] >  cutoff]

xgb = XGBRegressor(n_estimators=300, max_depth=4, learning_rate=0.05)
xgb.fit(tr[cols], tr['sales'])

te = te.copy()
te['xgb_pred'] = xgb.predict(te[cols]).clip(min=0)

In [29]:
def rmse(a, p): return np.sqrt(np.mean((a - p) ** 2))

def mape_nonzero(a, p):
    mask = a > 0
    return np.mean(np.abs((a[mask] - p[mask]) / a[mask])) * 100

cmp = te.merge(prophet_test_pred, left_on=['sub_category', 'date'],
               right_on=['sub_category', 'ds'])

print('Prophet  RMSE:', rmse(cmp['sales'], cmp['yhat']),
      ' MAPE:', mape_nonzero(cmp['sales'].values, cmp['yhat'].values))
print('XGBoost  RMSE:', rmse(cmp['sales'], cmp['xgb_pred']),
      ' MAPE:', mape_nonzero(cmp['sales'].values, cmp['xgb_pred'].values))

Prophet  RMSE: 1708.3065891139588  MAPE: 385.2486050903759
XGBoost  RMSE: 1765.7164118378048  MAPE: 305.31710538945


In [30]:
# 1. WAPE: total error divided by total sales (not hurt by tiny weeks)
def wape(a, p):
    return np.abs(a - p).sum() / a.sum() * 100

print('Prophet  WAPE:', wape(cmp['sales'].values, cmp['yhat'].values))
print('XGBoost  WAPE:', wape(cmp['sales'].values, cmp['xgb_pred'].values))

# 2. Simple baseline: "next week = last week" (your models should beat this)
base = te['lag1'].values
print('Baseline RMSE:', rmse(te['sales'].values, base))
print('Baseline WAPE:', wape(te['sales'].values, base))

Prophet  WAPE: 70.44045141017905
XGBoost  WAPE: 73.89026035117018
Baseline RMSE: 2386.802369397561
Baseline WAPE: 97.43260127436166


In [31]:
import os, json
import pandas as pd
from prophet import Prophet
from prophet.serialize import model_to_json

os.makedirs('outputs/models', exist_ok=True)

rows = []
for s in sorted(weekly['sub_category'].unique()):
    d = (weekly[weekly['sub_category'] == s][['date', 'sales']]
         .rename(columns={'date': 'ds', 'sales': 'y'}))

    m = Prophet()
    m.fit(d)

    # save the model
    fname = s.replace(' ', '_')
    with open(f'outputs/models/prophet_{fname}.json', 'w') as f:
        f.write(model_to_json(m))

    # predict next week
    f = m.predict(m.make_future_dataframe(periods=1, freq='W')).tail(1)
    rows.append({
        'sub_category': s,
        'week_ending': f['ds'].iloc[0].date(),
        'predicted_sales': max(f['yhat'].iloc[0], 0),
        'lower_estimate': max(f['yhat_lower'].iloc[0], 0),
        'upper_estimate': max(f['yhat_upper'].iloc[0], 0),
    })

forecast = pd.DataFrame(rows).round(2)
forecast.to_csv('outputs/forecast_next_week.csv', index=False)

# keep the weekly data too, so models can be refreshed later
weekly.to_csv('outputs/weekly_sales.csv', index=False)
forecast

10:42:21 - cmdstanpy - INFO - Chain [1] start processing
10:42:21 - cmdstanpy - INFO - Chain [1] done processing
10:42:21 - cmdstanpy - INFO - Chain [1] start processing
10:42:21 - cmdstanpy - INFO - Chain [1] done processing
10:42:21 - cmdstanpy - INFO - Chain [1] start processing
10:42:21 - cmdstanpy - INFO - Chain [1] done processing
10:42:21 - cmdstanpy - INFO - Chain [1] start processing
10:42:22 - cmdstanpy - INFO - Chain [1] done processing
10:42:22 - cmdstanpy - INFO - Chain [1] start processing
10:42:22 - cmdstanpy - INFO - Chain [1] done processing
10:42:22 - cmdstanpy - INFO - Chain [1] start processing
10:42:22 - cmdstanpy - INFO - Chain [1] done processing
10:42:22 - cmdstanpy - INFO - Chain [1] start processing
10:42:22 - cmdstanpy - INFO - Chain [1] done processing
10:42:22 - cmdstanpy - INFO - Chain [1] start processing
10:42:22 - cmdstanpy - INFO - Chain [1] done processing
10:42:22 - cmdstanpy - INFO - Chain [1] start processing
10:42:22 - cmdstanpy - INFO - Chain [1]

,sub_category,week_ending,predicted_sales,lower_estimate,upper_estimate
0,Accessories,2019-01-06,1068.68,116.13,2009.95
1,Appliances,2019-01-06,848.12,98.94,1612.06
2,Art,2019-01-06,122.39,0.00,280.77
3,Binders,2019-01-06,1054.08,0.00,3021.49
4,Bookcases,2019-01-06,1052.71,0.00,2136.41
5,Chairs,2019-01-06,1749.51,0.00,3581.42
6,Copiers,2019-01-06,1316.96,0.00,3992.23
7,Envelopes,2018-12-30,72.08,0.00,206.90
8,Fasteners,2019-01-06,23.67,4.93,44.63
9,Furnishings,2019-01-06,649.20,122.09,1156.18


In [32]:
cat_map = df[['Sub-Category', 'Category']].drop_duplicates().set_index('Sub-Category')['Category']
forecast['category'] = forecast['sub_category'].map(cat_map)
forecast.to_csv('outputs/forecast_next_week.csv', index=False)